# Environment configuration — landing/vlp
Credentials are external. Set PIPELINE_ENV and job variables generated from config/environments and config/secrets.
This repository revision changes configuration handling, not the source ingestion/write algorithms.


# NB_landing.vlp.order_detail
API → Landing JSON, giữ cấu trúc cell và cấu hình kết nối theo notebook mẫu HRM.
Landing: `s3://landing-bucket/vlp/order_detail/YYYYMMDD/`. Bảng: `bronze.vlp.order_detail`.


## Import, cấu hình và chạy
1. CP4D → Projects → DataLakehouse → Assets → New asset → Notebook → Local file.
2. Landing dùng Python với requests/boto3 (không cần Spark). Bronze dùng Python 3.11 with Spark và connector watsonx.data như notebook HRM mẫu. Restart kernel và Run All sau khi import.
3. Cấu hình S3/metastore/Spark giữ từ mẫu. API mặc định UAT và test secret trong spec. Không dùng sign cố định của curl: chữ ký được tính lại theo toàn bộ tham số bên trong, kể cả số trang.
4. Daily: bỏ các biến VLP_DATE_FROM/VLP_DATE_TO, ETL_Date mặc định T−1 Việt Nam; request theo ngày tạo từ 00:00 đến trước 00:00 hôm sau. Spec mô tả createTimeEnd inclusive nên gửi cuối ngày = cận trên trừ 1 đơn vị.
5. Initial: truyền ETL_Date=ngày lưu tập khởi tạo, VLP_DATE_FROM=2020-01-01, VLP_DATE_TO=ngày kết thúc exclusive. Ví dụ ETL_Date=2026-10-21, VLP_DATE_FROM=2020-01-01, VLP_DATE_TO=2026-10-22. Một tập init được lưu vào partition batch, không tái dựng partition lịch sử từng ngày.
6. Landing lấy và kiểm tra đủ mọi trang trong thư mục tạm local; xóa file cũ đúng vlp/order_detail/YYYYMMDD/, upload trang mới, chốt manifest cuối cùng. Lỗi API trước bước xóa giữ tập cũ; lỗi ghi file không có manifest hoàn tất. Batch rỗng hợp lệ vẫn thay thế ngày.
7. Bronze dùng cùng ETL_Date, tự đọc manifest; MANIFEST_PATH và LANDING_RUN_ID là tùy chọn để khóa đúng tập đầu vào. Giữ nguyên nested data dưới dạng JSON string trong cột tương ứng và raw_record; không explode parcel/track làm nhân bản order. Ghi atomic overwrite partition ngày; log tại bronze.pipeline.log_pipeline.
8. Chỉ chạy Bronze sau khi Landing SUCCESS. Không chạy lại Landing cùng ngày trong khi Bronze đang đọc. Chỉ một job xử lý cùng nguồn/object/ngày; tổng tốc độ các job cùng gateway phải dưới giới hạn 10 request/giây. Mặc định một job gọi 5 request/giây.

## Phạm vi và điểm cần kiểm tra trên UAT
- Spec chỉ có createTimeStart/createTimeEnd, chưa có bộ lọc cập nhật gmtModified. Daily hiện lấy đơn được tạo trong ngày; không bảo đảm bắt thay đổi/tracking mới của các đơn cũ. Cần API lọc cập nhật hoặc cơ chế quét lại/đối soát khi bổ sung yêu cầu đó.
- createTimeStart/createTimeEnd có kiểu Long nhưng spec chưa ghi đơn vị. Notebook mặc định epoch milliseconds, cho phép VLP_TIME_UNIT=seconds. Kiểm tra với một đơn biết gmtCreate và thời gian đúng trước khi lập lịch; notebook kiểm tra mọi gmtCreate trong response thuộc cửa sổ request.
- Phân trang dùng currentPage/pageSize, response data.paging.totalCount. Tổng thay đổi, lặp trang, trùng orderCode hoặc thiếu trang đều dừng trước khi thay thế dữ liệu ngày. Pagination khớp không chứng minh nguồn có snapshot nhất quán giữa các trang.
- Mỗi trang giữ trong RAM; toàn bộ dữ liệu tạm/khóa chống trùng lưu trên disk driver. Bronze stage NDJSON lên S3 để Spark đọc phân tán; xóa stage sau khi hoàn tất/lỗi. Cần đủ disk tạm theo volume ngày/init.
- Chưa gọi API hoặc chạy Spark/metastore thực tế; việc giữ cấu hình mẫu không tự sửa lỗi metastore đã gặp ở HRM.


In [ ]:
# Landing không cần khởi tạo hoặc dừng Spark.


In [ ]:
import os, sys, json, time, logging, re, uuid, hashlib, tempfile, sqlite3
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import boto3
from botocore.config import Config
import requests


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout, force=True)
logger = logging.getLogger("vlp_ingest")

In [ ]:
start_date = datetime.now(timezone.utc)
status, log_message, record_count, df = "Fail", None, 0, None

In [ ]:
# Environment configuration is injected by CP4D/job environment variables.
# Generate a job env file with scripts/build_job_env.py. Never embed credentials here.
def _need_env(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu biến môi trường: " + name)
    return value

PIPELINE_ENV = _need_env("PIPELINE_ENV")
if PIPELINE_ENV not in ("dev", "uat", "prod"):
    raise ValueError("PIPELINE_ENV phải là dev, uat hoặc prod")

# 1. Lấy các biến môi trường
now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
yesterday = now - timedelta(days=1)

# 3. Định dạng lại thành chuỗi YYYYMMDD
yesterday = yesterday.strftime("%Y-%m-%d")

#S3
S3_ENDPOINT = _need_env("S3_ENDPOINT")
ACCESS_KEY = _need_env("ACCESS_KEY")
SECRET_KEY = _need_env("SECRET_KEY")
BUCKET_NAME = _need_env("BUCKET_NAME")

#Metastore
WXD_HMS_USERNAME = os.getenv("WXD_HMS_USERNAME", "")
WXD_HMS_PASSWORD = os.getenv("WXD_HMS_PASSWORD", "")
ICEBERG_CATALOG1 = os.getenv("ICEBERG_CATALOG1", "bronze")
BUCKET_NAME1 = os.getenv("BUCKET_NAME1", "")
ACCESS_KEY1 = os.getenv("ACCESS_KEY1") or ACCESS_KEY
SECRET_KEY1 = os.getenv("SECRET_KEY1") or SECRET_KEY

HIVEMETASTORE_HOST = os.getenv("HIVEMETASTORE_HOST", "")

HIVE_TRUSTSTORE_PATH = os.getenv("HIVE_TRUSTSTORE_PATH", "")
HIVE_TRUSTSTORE_PASSWORD = os.getenv("HIVE_TRUSTSTORE_PASSWORD", "")

WXD_INSTANCE_ID = os.getenv("WXD_INSTANCE_ID", "")
HADOOP_AMS_URL = os.getenv("HADOOP_AMS_URL", "")
HADOOP_CAS_ENDPOINT = os.getenv("HADOOP_CAS_ENDPOINT", "")
HADOOP_CPG_ENDPOINT = os.getenv("HADOOP_CPG_ENDPOINT", "")


TABLE_NAME=os.getenv("TABLE_NAME","bronze.vlp.order_detail")
SOURCE_NAME=os.getenv("SOURCE_NAME","vlp.api.order_detail")
STEP='api_2_landing'


ETL_Date = os.getenv("ETL_Date",yesterday)
TZ = ZoneInfo("Asia/Ho_Chi_Minh")
BATCH_DATE = date.fromisoformat(ETL_Date)
if ETL_Date != BATCH_DATE.isoformat(): raise ValueError("ETL_Date phải đúng YYYY-MM-DD")
DAY_PREFIX = f"vlp/order_detail/{BATCH_DATE.strftime('%Y%m%d')}"
LOG_PREFIX = f"vlp/order_detail/_logs/{BATCH_DATE.strftime('%Y%m%d')}"
LOG_TABLE = os.getenv("LOG_TABLE", "bronze.pipeline.log_pipeline")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", ICEBERG_CATALOG1): raise ValueError("Catalog không hợp lệ")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", TABLE_NAME):
    raise ValueError("TABLE_NAME phải là catalog.schema.table")
RUN_ID = uuid.uuid4().hex

# API 1.45 từ spec liên kết tại dòng VLP trong bảng khảo sát.
VLP_URL = _need_env("VLP_URL")
# Test secret trong spec; environment VLP_SIGN_SECRET được ưu tiên.
_need_env("VLP_SIGN_SECRET")
PAGE_SIZE = int(os.getenv("VLP_PAGE_SIZE", "20"))
MAX_PAGES = int(os.getenv("VLP_MAX_PAGES", "100000"))
REQUESTS_PER_SECOND = float(os.getenv("VLP_REQUESTS_PER_SECOND", "5"))
if PAGE_SIZE <= 0 or MAX_PAGES <= 0 or not 0 < REQUESTS_PER_SECOND <= 10:
    raise ValueError("Page size/max pages phải dương; request rate trong (0,10]")
# Mặc định theo ETL_Date. Muốn init: truyền VLP_DATE_FROM=2020-01-01,
# VLP_DATE_TO là ngày kết thúc exclusive; ETL_Date là partition lưu tập init.
# os.environ["ETL_Date"] phải được đặt TRƯỚC cell cấu hình này.
# os.environ["VLP_DATE_FROM"] = "2020-01-01"
# os.environ["VLP_DATE_TO"] = "2026-10-22"
# VLP_TIME_UNIT mặc định milliseconds; spec ghi Long, chưa chỉ rõ đơn vị.
# VLP_ORDER_CODE tùy chọn để test một đơn, mặc định không lọc orderCode.


In [ ]:
def init_spark():
    spark = SparkSession.builder \
        .appName(f"import_table") \
        .config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}","org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.type", "hive") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.uri",
                f"{HIVEMETASTORE_HOST}?catalog={ICEBERG_CATALOG1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.warehouse",
                f"s3a://{BUCKET_NAME1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.username",
                WXD_HMS_USERNAME) \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.password",
                WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.uris",HIVEMETASTORE_HOST) \
        .config("spark.hive.metastore.use.SSL", "true") \
        .config("spark.hive.metastore.truststore.type", "JKS") \
        .config("spark.hive.metastore.truststore.path", HIVE_TRUSTSTORE_PATH) \
        .config("spark.hive.metastore.truststore.password", HIVE_TRUSTSTORE_PASSWORD) \
        .config("spark.hive.metastore.client.auth.mode", "PLAIN") \
        .config("spark.hive.metastore.client.plain.username", WXD_HMS_USERNAME) \
        .config("spark.hive.metastore.client.plain.password", WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.client.thrift.transport.mode", "http") \
        .config("spark.hadoop.wxd.ams.url", HADOOP_AMS_URL) \
        .config("spark.hadoop.wxd.cas.endpoint", HADOOP_CAS_ENDPOINT) \
        .config("spark.hadoop.wxd.cpg.endpoint", HADOOP_CPG_ENDPOINT) \
        .config("spark.hadoop.wxd.instanceId", WXD_INSTANCE_ID) \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        \
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.access.key", ACCESS_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.secret.key", SECRET_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.connection.ssl.enabled", "true") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.access.key", ACCESS_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.secret.key", SECRET_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.connection.ssl.enabled", "true") \
        \
        .config("spark.hadoop.iceberg.engine.hive.lock-enabled", "false") \
        .config("spark.sql.catalogImplementation", "hive") \
        .getOrCreate()


    spark.sparkContext.setLogLevel("WARN")
    return spark

In [ ]:
def required(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu cấu hình: " + name)
    return value

def pack(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")

def init_s3():
    return boto3.client("s3", endpoint_url=S3_ENDPOINT,
        aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4", connect_timeout=15, read_timeout=120,
                      retries={"max_attempts": 3, "mode": "standard"},
                      s3={"addressing_style": "path"}))

def put_json(key, obj):
    blob = pack(obj)
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=blob, ContentType="application/json")
    return hashlib.sha256(blob).hexdigest()

def get_json(key):
    return json.loads(s3.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read())

def delete_prefix(prefix):
    # Chỉ cho phép đúng partition ngày hoặc thư mục stage của run hiện tại.
    allowed = {DAY_PREFIX + "/", f"vlp/order_detail/_bronze_stage/{RUN_ID}/"}
    if prefix not in allowed:
        raise ValueError("Từ chối xóa ngoài phạm vi job")
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET_NAME, Prefix=prefix):
        keys.extend(x["Key"] for x in page.get("Contents", []))
    # Xóa manifest trước để không chốt tập file đang thay thế.
    manifest_key = prefix + "manifest.json"
    if manifest_key in keys:
        keys.remove(manifest_key)
        keys.insert(0, manifest_key)
        result = s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": [{"Key": manifest_key}]})
        if result.get("Errors"):
            raise RuntimeError("Xóa manifest không thành công")
        keys.remove(manifest_key)
    for i in range(0, len(keys), 1000):
        result = s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": [{"Key": k} for k in keys[i:i+1000]]})
        if result.get("Errors"):
            raise RuntimeError("Xóa file S3 không thành công")

def error_summary(exc):
    if isinstance(exc, ValueError) and not isinstance(exc, json.JSONDecodeError):
        return str(exc)
    if isinstance(exc, VlpApiError):
        return str(exc)
    response = getattr(exc, "response", None)
    if isinstance(response, dict):
        return "S3 error: " + str(response.get("Error", {}).get("Code", "unknown"))
    return type(exc).__name__ + ": kiểm tra kết nối/runtime; không in credentials hoặc dữ liệu đơn hàng."

class VlpApiError(RuntimeError):
    pass

def decode_page(payload, page, requested_size):
    if not isinstance(payload, dict):
        raise ValueError("Response VLP không phải object")
    if payload.get("success") is not True:
        # Không log response/errorMessage vì có thể chứa request hoặc dữ liệu cá nhân.
        code = re.sub(r"[^A-Za-z0-9_.-]", "", str(payload.get("errorCode", "unknown")))[:100]
        trace = re.sub(r"[^A-Za-z0-9_.-]", "", str(payload.get("traceId", "unknown")))[:100]
        raise VlpApiError(f"VLP success != true; errorCode={code}; traceId={trace}")
    data = payload.get("data")
    if not isinstance(data, dict) or not isinstance(data.get("paging"), dict):
        raise ValueError("Thiếu data.paging")
    rows, paging = data.get("tableData"), data["paging"]
    if not isinstance(rows, list):
        raise ValueError("data.tableData không phải danh sách")
    for k in ("currentPage", "pageSize", "totalCount"):
        if type(paging.get(k)) is not int:
            raise ValueError("Metadata phân trang không phải integer: " + k)
    if paging["currentPage"] != page or paging["pageSize"] <= 0 or paging["totalCount"] < 0:
        raise ValueError("Metadata phân trang không hợp lệ")
    if paging["pageSize"] != requested_size or len(rows) > paging["pageSize"]:
        raise ValueError("pageSize trả về khác request hoặc số dòng vượt pageSize; điều chỉnh VLP_PAGE_SIZE theo server")
    for row in rows:
        if not isinstance(row, dict) or not isinstance(row.get("orderCode"), str) or not row["orderCode"]:
            raise ValueError("Bản ghi thiếu orderCode")
    return rows, paging

def validate_manifest(manifest):
    if (manifest.get("manifest_version") != 1 or manifest.get("source") != "vlp"
        or manifest.get("object") != "order_detail" or manifest.get("data_date") != ETL_Date
        or manifest.get("complete") is not True):
        raise ValueError("Manifest sai nguồn/object/ngày hoặc chưa chốt")
    if not re.fullmatch(r"[0-9a-f]{32}", manifest.get("run_id", "")):
        raise ValueError("Manifest run_id không hợp lệ")
    if type(manifest.get("count")) is not int or manifest["count"] < 0:
        raise ValueError("Manifest count không hợp lệ")
    files = manifest.get("files")
    if not isinstance(files, list) or not files:
        raise ValueError("Manifest thiếu danh sách file")
    for i, item in enumerate(files, 1):
        if item.get("key") != f"{DAY_PREFIX}/page-{i:06d}.json":
            raise ValueError("Manifest sai đường dẫn/thứ tự trang")
        if not re.fullmatch(r"[0-9a-f]{64}", item.get("sha256", "")):
            raise ValueError("Manifest thiếu checksum")
        if type(item.get("count")) is not int or item["count"] < 0:
            raise ValueError("Manifest page count không hợp lệ")
    if sum(x["count"] for x in files) != manifest["count"]:
        raise ValueError("Tổng số dòng manifest không khớp")

def seen_order(connection, key):
    try:
        connection.execute("INSERT INTO seen VALUES (?)", (key,))
    except sqlite3.IntegrityError:
        raise ValueError("orderCode trùng giữa/trong các trang; không chốt dữ liệu") from None


In [ ]:
def java_value(value):
    if value is None: return "null"
    if value is True: return "true"
    if value is False: return "false"
    if isinstance(value, (dict, list)): return pack(value).decode("utf-8")
    return str(value)

def flatten_array(key, values):
    parts = []
    for value in values:
        if isinstance(value, list): parts.append(flatten_array(key, value))
        elif isinstance(value, dict): parts.append(flatten_object(value))
        else: parts.append(java_value(value))
    return (key + "=" if key else "") + ",".join(parts)

def flatten_object(obj):
    parts = []
    # Java Collections.sort(String) compares UTF-16 code units.
    for key in sorted(obj, key=lambda x: x.encode("utf-16-be")):
        value = obj[key]
        if isinstance(value, dict): parts.append(flatten_object(value))
        elif isinstance(value, list): parts.append(flatten_array(key, value))
        else: parts.append(key + "=" + java_value(value))
    return "&".join(parts)

def calculate_sign(inner, secret):
    flat = flatten_array("", inner) if isinstance(inner, list) else flatten_object(inner)
    return hashlib.md5((flat + "&" + secret).encode("utf-8")).hexdigest()

def query_window():
    start_date = date.fromisoformat(os.getenv("VLP_DATE_FROM") or ETL_Date)
    end_date = date.fromisoformat(os.getenv("VLP_DATE_TO") or (BATCH_DATE + timedelta(days=1)).isoformat())
    if end_date <= start_date:
        raise ValueError("VLP_DATE_TO phải lớn hơn VLP_DATE_FROM")
    start = datetime.combine(start_date, datetime.min.time(), tzinfo=TZ)
    end = datetime.combine(end_date, datetime.min.time(), tzinfo=TZ)
    unit = os.getenv("VLP_TIME_UNIT", "milliseconds")
    if unit not in ("milliseconds", "seconds"):
        raise ValueError("VLP_TIME_UNIT phải là milliseconds hoặc seconds")
    factor = 1000 if unit == "milliseconds" else 1
    return start, end, int(start.timestamp()) * factor, int(end.timestamp()) * factor - 1, unit

def build_inner(page):
    start, end, lower, upper, unit = query_window()
    inner = {"currentPage": page, "pageSize": PAGE_SIZE,
             "createTimeStart": lower, "createTimeEnd": upper,
             "isQueryFeeList": True, "isQueryOrderParcel": True, "isQueryTrack": True}
    order_code = os.getenv("VLP_ORDER_CODE", "").strip()
    if order_code: inner["orderCode"] = order_code
    return inner

_last_request_time = 0.0
def fetch_page(session, inner):
    global _last_request_time
    secret = required("VLP_SIGN_SECRET")
    envelope = {"appCode": "VLP", "appApiCode": "vnp.eoms.order.page.detail.sub",
                "schemeCode": "VLP-EOMS", "body": pack(inner).decode("utf-8")}
    headers = {"Content-Type": "application/json", "sign": calculate_sign(inner, secret)}
    parsed = urlparse(VLP_URL)
    if parsed.scheme != "https" or not parsed.netloc or parsed.username or parsed.query or parsed.fragment:
        raise ValueError("VLP_URL phải là HTTPS endpoint, không có credentials/query")
    for attempt in range(4):
        pause = 1.0 / REQUESTS_PER_SECOND - (time.monotonic() - _last_request_time)
        if pause > 0: time.sleep(pause)
        _last_request_time = time.monotonic()
        try:
            response = session.post(VLP_URL, headers=headers, json=envelope,
                timeout=(15, 120), verify=os.getenv("VLP_CA_BUNDLE") or True, allow_redirects=False)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == 3: raise VlpApiError("VLP timeout/kết nối thất bại sau retry") from None
            time.sleep(2 ** attempt)
            continue
        logger.info("VLP page=%s HTTP=%s", inner["currentPage"], response.status_code)
        if response.status_code in (429, 500, 502, 503, 504) and attempt < 3:
            time.sleep(2 ** attempt)
            continue
        if response.status_code != 200:
            raise VlpApiError(f"VLP HTTP {response.status_code}")
        try: return response.json()
        except ValueError: raise VlpApiError("VLP không trả JSON") from None
    raise VlpApiError("Không lấy được trang VLP")

def check_create_date(row, start, end):
    # Kiểm tra server thực sự áp dụng cửa sổ ngày tạo, tránh chốt sai epoch unit/filter.
    raw = row.get("gmtCreate")
    if not isinstance(raw, str) or not raw:
        raise ValueError("Thiếu gmtCreate để đối soát bộ lọc ngày")
    try: stamp = datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except ValueError: raise ValueError("gmtCreate không phải ISO datetime như response mẫu") from None
    if stamp.tzinfo is None: stamp = stamp.replace(tzinfo=TZ)
    if not start <= stamp < end:
        raise ValueError("gmtCreate nằm ngoài khoảng request; kiểm tra timezone, epoch unit và bộ lọc API")

def extract():
    if os.getenv("VLP_ORDER_CODE", "").strip():
        raise ValueError("VLP_ORDER_CODE chỉ dùng với probe_api(); bỏ bộ lọc đơn lẻ trước khi thay thế partition ngày")
    start, end, lower, upper, unit = query_window()
    count, expected, fingerprints, files = 0, None, set(), []
    logger.info("API createTime: %s -> %s (cận trên exclusive); request Long unit=%s", start.isoformat(), end.isoformat(), unit)
    with tempfile.TemporaryDirectory(prefix="vlp_landing_") as tmp:
        with sqlite3.connect(os.path.join(tmp, "seen.sqlite")) as seen:
            seen.execute("CREATE TABLE seen (order_code TEXT PRIMARY KEY)")
            with requests.Session() as session:
                for page in range(1, MAX_PAGES + 1):
                    payload = fetch_page(session, build_inner(page))
                    rows, paging = decode_page(payload, page, PAGE_SIZE)
                    if expected is None: expected = paging["totalCount"]
                    elif expected != paging["totalCount"]:
                        raise ValueError("totalCount thay đổi giữa các trang; không chốt tập thiếu nhất quán")
                    digest_rows = hashlib.sha256(pack(rows)).hexdigest()
                    if rows and digest_rows in fingerprints: raise ValueError("API trả lặp trang")
                    fingerprints.add(digest_rows)
                    for row in rows:
                        seen_order(seen, row["orderCode"])
                        check_create_date(row, start, end)
                    blob = pack(payload)
                    name = f"page-{page:06d}.json"
                    with open(os.path.join(tmp, name), "wb") as out: out.write(blob)
                    files.append({"key": f"{DAY_PREFIX}/{name}", "count": len(rows),
                                  "sha256": hashlib.sha256(blob).hexdigest()})
                    count += len(rows)
                    logger.info("VLP page=%s rows=%s collected=%s total=%s", page, len(rows), count, expected)
                    if count > expected: raise ValueError("Số dòng vượt totalCount")
                    if count == expected: break
                    if not rows or len(rows) < paging["pageSize"]:
                        raise ValueError("Trang rỗng/thiếu dòng trước khi đạt totalCount")
                else: raise ValueError("Vượt VLP_MAX_PAGES; không chốt Landing")
        # Không xóa tập ngày cũ trước khi lấy và kiểm tra đủ dữ liệu API.
        delete_prefix(DAY_PREFIX + "/")
        for item in files:
            s3.upload_file(os.path.join(tmp, item["key"].rsplit("/", 1)[-1]), BUCKET_NAME, item["key"],
                           ExtraArgs={"ContentType": "application/json"})
        manifest = {"manifest_version": 1, "source": "vlp", "object": "order_detail", "data_date": ETL_Date,
            "run_id": RUN_ID, "ingest_time": datetime.now(timezone.utc).isoformat(), "count": count,
            "files": files, "complete": True, "page_size": PAGE_SIZE,
            "window": {"field": "gmtCreate", "from": start.isoformat(), "to_exclusive": end.isoformat(),
                       "createTimeStart": lower, "createTimeEnd": upper, "unit": unit},
            "appApiCode": "vnp.eoms.order.page.detail.sub"}
        validate_manifest(manifest)
        put_json(DAY_PREFIX + "/manifest.json", manifest)
        return manifest

def probe_api():
    # Gọi một trang để chẩn đoán; không ghi/xóa Landing và không in PII.
    with requests.Session() as session:
        payload = fetch_page(session, build_inner(1))
    rows, paging = decode_page(payload, 1, PAGE_SIZE)
    print("success=True; paging=", paging, "; rows=", len(rows))
    print("window=", {k: v for k,v in build_inner(1).items() if k in ("createTimeStart", "createTimeEnd")})
    if rows: print("Các trường:", list(rows[0]))
    return payload


## Test chữ ký và gọi thử
Cell này kiểm tra chữ ký theo ví dụ tài liệu. Để gọi một trang API mà không ghi/xóa Landing, chạy `probe_api()` trong cell riêng.

In [ ]:
# Synthetic signing fixture; not a DEV credential.
sample_body = {"currentPage":1,"pageSize":10,"orderCode":"VNPE260715000000104","isQueryOrderParcel":True,"isQueryTrack":True}
assert calculate_sign(sample_body, "example-only-not-a-credential") == "90d7f673afa97e2ce7972b9f1cdca644"
print("Signature example: OK")


## Chạy API → Landing

In [ ]:
globals().pop("LANDING_MANIFEST", None)
RUN_ID = uuid.uuid4().hex
start_date = datetime.now(timezone.utc)
status, log_message, record_count = "Fail", None, 0
s3 = None
logger.info("START VLP API -> Landing date=%s run_id=%s", ETL_Date, RUN_ID)
try:
    s3 = init_s3()
    LANDING_MANIFEST = extract()
    record_count, status = LANDING_MANIFEST["count"], "Success"
    logger.info("SUCCESS rows=%s pages=%s", record_count, len(LANDING_MANIFEST["files"]))
    print("MANIFEST_PATH =", f"s3://{BUCKET_NAME}/{DAY_PREFIX}/manifest.json", flush=True)
    print("LANDING_RUN_ID =", RUN_ID, flush=True)
except Exception as exc:
    log_message = error_summary(exc)
    logger.error("FAIL %s", log_message)
    raise RuntimeError(log_message) from None
finally:
    if s3 is not None:
        try:
            put_json(f"{LOG_PREFIX}/landing-{RUN_ID}.json", {
                "source": "vlp", "object": "order_detail", "data_date": ETL_Date,
                "run_id": RUN_ID, "status": status, "record_count": record_count,
                "start_date": start_date.isoformat(), "end_date": datetime.now(timezone.utc).isoformat(),
                "error": log_message})
        except Exception as log_exc: logger.error("Ghi log S3 thất bại: %s", type(log_exc).__name__)


In [ ]:
if "LANDING_MANIFEST" not in globals():
    print("Chưa có lần chạy Landing thành công")
else:
    saved = get_json(DAY_PREFIX + "/manifest.json")
    validate_manifest(saved)
    print(json.dumps({k:saved[k] for k in ("data_date","run_id","count","complete","window")}, ensure_ascii=False, indent=2))
    print("Số trang:", len(saved["files"]))


## Nguồn thiết kế
- Bảng khảo sát: https://docs.google.com/spreadsheets/d/1k3VU08FAk2gbAm-cUYudu0pMfcZZWGG4/edit?gid=168101360#gid=168101360 — dòng VLP, spec liên kết tại cột R.
- API spec: https://docs.google.com/document/d/1MHpi_8CDwkOmR8MEQIkl0QkiS1RXxf8a/edit — API 1.45, vnp.eoms.order.page.detail.sub.
